<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/basic_pruning_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#OptiPFair Notebook Series – Example: Basic Pruning (MLP)

![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)


This notebook demonstrates how to use [OptiPFair](https://github.com/peremartra/optipfair) for structured pruning of transformer models with GLU-based MLP layers.  
The example covers both percentage-based and expansion-rate-based pruning strategies.

##Recommended Environment

- **Platform**: [Google Colab](https://colab.research.google.com)  
- **Hardware**: GPU runtime (recommended: T4 or better for 1B–3B models)  
- **Dependencies**: Installed automatically in the first cell (optipfair, transformers, torch)

##by Pere Martra.

- [LinkedIn](https://www.linkedin.com/in/pere-martra)  
- [GitHub](https://github.com/peremartra)  
- [X / Twitter](https://x.com/peremartra)

---

> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!
---
If you want your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.


# Basic OptiPFair Pruning Example

This notebook demonstrates how to use OptiPFair for structured pruning of language models.
OptiPFair focuses on pruning MLP layers with GLU (Gated Linear Unit) architecture,
which is commonly found in modern models like LLaMA, Gemma, Mistral, and others.

Author: Pere Martra

Designed for Google Colab - GPU runtime recommended

---
## Installation and Setup


In [1]:
!pip install -q transformers optipfair torch


## Import Libraries and Check GPU


In [2]:
import torch
import os
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from optipfair import prune_model

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

2025-09-09 14:51:34.603037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757429494.618312    2020 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757429494.623110    2020 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-09 14:51:34.638693: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda
GPU: NVIDIA A10G
GPU Memory: 22.0 GB


## Configuration


In [13]:
# List of models to test - you can add more GLU-compatible models here
# Note: For Colab, stick to smaller models due to memory constraints
MODELS_TO_TEST = [
    #"meta-llama/Llama-3.2-1B",
    "Qwen/Qwen2.5-7B-Instruct"
    #"google/gemma-2-2b",  # Uncomment if you have enough GPU memory
    # Add more models here as needed
]

# Pruning configuration - modify these values as needed
PRUNING_PERCENTAGE = 20  # Percentage of neurons to remove (0-100)
TARGET_EXPANSION_RATE = 200  # Alternative: target expansion rate (e.g., 200% instead of ~400%)

# Test prompts for evaluation
TEST_PROMPTS = [
    "Paris is the capital of",
    "The theory of relativity states that",
    "Machine learning is a field of",
]

print("Configuration set successfully!")
print(f"Models to test: {MODELS_TO_TEST}")
print(f"Pruning percentage: {PRUNING_PERCENTAGE}%")
print(f"Target expansion rate: {TARGET_EXPANSION_RATE}%")


Configuration set successfully!
Models to test: ['Qwen/Qwen2.5-7B-Instruct']
Pruning percentage: 20%
Target expansion rate: 200%


## Introduction to Structured Pruning

---
This example demonstrates structured pruning of MLP layers in transformer models.

Structured pruning removes entire neurons while maintaining model architecture, resulting in actual speedup and memory reduction during inference.


## Utility Functions


In [14]:
def count_parameters(model):
    """Count total parameters in model"""
    return sum(p.numel() for p in model.parameters())

def test_model_generation(model, tokenizer, prompt, max_length=50):
    """Test text generation with the model"""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False,
            num_beams=3,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def print_model_info(model, model_name, stage=""):
    """Print basic model information"""
    param_count = count_parameters(model)
    print(f"{stage} Model: {model_name}")
    print(f"Parameters: {param_count:,}")
    return param_count

def cleanup_memory():
    """Clean up GPU memory - important for Colab"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Utility functions defined successfully!")

Utility functions defined successfully!



## OptiPFair Parameters Explanation
• model: The model to be pruned

• pruning_type: Type of pruning (currently only 'MLP_GLU' supported)

• neuron_selection_method: Method to calculate neuron importance:
  - 'MAW': Maximum Absolute Weight (recommended for most models)
  - 'VOW': Variance of Weights (alternative method)
  - 'PON': Product of Norms (alternative method)
  
• pruning_percentage: Percentage of neurons to remove (0-100)

• expansion_rate: Alternative to pruning_percentage - target expansion rate

• show_progress: Display progress bar during pruning

• return_stats: Return detailed statistics about pruning

## Example 1 - Pruning by Percentage Function


In [15]:
def example_pruning_by_percentage(model, tokenizer, model_name):
    """Example of pruning by neuron percentage"""
    print(f"=== Example 1: Pruning {model_name} by {PRUNING_PERCENTAGE}% ===")

    # Get original model info
    original_params = print_model_info(model, model_name, "Original")

    # Test original model
    print("\n--- Original Model Generation ---")
    for prompt in TEST_PROMPTS[:2]:  # Test first 2 prompts
        generated = test_model_generation(model, tokenizer, prompt)
        print(f"Prompt: '{prompt}'")
        print(f"Generated: {generated}")
        print()

    # Apply pruning by percentage
    pruned_model, stats = prune_model(
        model=model,
        pruning_type="MLP_GLU",
        neuron_selection_method="MAW",  # Change to "VOW" or "PON" to try other methods
        pruning_percentage=PRUNING_PERCENTAGE,
        show_progress=True,
        return_stats=True
    )

    # Print pruning statistics
    print("\n--- Pruning Results ---")
    print(f"Original parameters: {stats['original_parameters']:,}")
    print(f"Pruned parameters: {stats['pruned_parameters']:,}")
    print(f"Reduction: {stats['reduction']:,} parameters ({stats['percentage_reduction']:.2f}%)")
    print(f"Final expansion rate: {stats['expansion_rate']:.2f}%")

    # Test pruned model
    print("\n--- Pruned Model Generation ---")
    for prompt in TEST_PROMPTS[:2]:
        generated = test_model_generation(pruned_model, tokenizer, prompt)
        print(f"Prompt: '{prompt}'")
        print(f"Generated: {generated}")
        print()

    return pruned_model, stats

print("Example 1 function defined!")

Example 1 function defined!



## Example 2 - Pruning by Expansion Rate Function


In [16]:
def example_pruning_by_expansion_rate(model, tokenizer, model_name):
    """Example of pruning by target expansion rate"""
    print(f"=== Example 2: Pruning {model_name} to {TARGET_EXPANSION_RATE}% expansion rate ===")

    # Get original model info
    original_params = print_model_info(model, model_name, "Original")

    # Apply pruning by expansion rate
    pruned_model, stats = prune_model(
        model=model,
        pruning_type="MLP_GLU",
        neuron_selection_method="MAW",
        pruning_percentage=None,  # Must be None when using expansion_rate
        expansion_rate=TARGET_EXPANSION_RATE,  # Target expansion rate instead of percentage
        show_progress=True,
        return_stats=True
    )

    # Print pruning statistics
    print("\n--- Pruning Results ---")
    print(f"Original parameters: {stats['original_parameters']:,}")
    print(f"Pruned parameters: {stats['pruned_parameters']:,}")
    print(f"Reduction: {stats['reduction']:,} parameters ({stats['percentage_reduction']:.2f}%)")
    print(f"Target expansion rate: {TARGET_EXPANSION_RATE}%")
    print(f"Actual expansion rate: {stats['expansion_rate']:.2f}%")

    # Test pruned model with one prompt
    print("\n--- Pruned Model Generation ---")
    prompt = TEST_PROMPTS[0]
    generated = test_model_generation(pruned_model, tokenizer, prompt)
    print(f"Prompt: '{prompt}'")
    print(f"Generated: {generated}")
    print()

    return pruned_model, stats

print("Example 2 function defined!")

Example 2 function defined!


## Run Example 1 - Pruning by Percentage


In [17]:
print("Starting Example 1: Pruning by Percentage")
print("=" * 50)

# Process the first model in the list
model_name = MODELS_TO_TEST[0]
print(f"Loading model: {model_name}")

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set pad token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Run Example 1
pruned_model_1, stats_1 = example_pruning_by_percentage(model, tokenizer, model_name)

# Store stats for summary
results = [{
    'model': model_name,
    'method': 'Percentage',
    'reduction': stats_1['percentage_reduction'],
    'expansion_rate': stats_1['expansion_rate']
}]

print(f"\nExample 1 completed! Reduction: {stats_1['percentage_reduction']:.2f}%")

Starting Example 1: Pruning by Percentage
Loading model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

...

Prompt: 'Paris is the capital of'
Generated: Paris is the capital of France and one of the most beautiful cities in the world. It is located on the Seine River in northern France. Paris has a population of about 2.2 million people, making it the second largest city in Europe

Prompt: 'The theory of relativity states that'
Generated: The theory of relativity states that the speed of light in a vacuum is the same for all observers, regardless of their motion relative to the light source. Suppose a spaceship is traveling at a constant velocity of $0.8c$ (where $



Pruning layers: 100%|██████████| 28/28 [00:29<00:00,  1.04s/it]
2025-09-09 15:15:34,076 - optipfair.pruning.mlp_glu - INFO - Updated model config: intermediate_size = 15156
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Pruning Results ---
Original parameters: 7,615,616,512
Pruned parameters: 6,475,216,384
Reduction: 1,140,400,128 parameters (14.97%)
Final expansion rate: 422.88%

--- Pruned Model Generation ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt: 'Paris is the capital of'
Generated: Paris is the capital of France. Paris is famous for its famous landmarks. Which of the following landmarks is not famous in Paris?
A. Eiffel Tower
B. Arc de Triomphe
C. Louvre Museum
D. Notre

Prompt: 'The theory of relativity states that'
Generated: The theory of relativity states that the speed of light in a vacuum is constant and independent of the reference frame in which it is measured. A consequence of this is that time runs slower at high speeds or high altitudes as compared to low speeds and


Example 1 completed! Reduction: 14.97%


## Run Example 2 - Pruning by Expansion Rate


In [18]:
print("Starting Example 2: Pruning by Expansion Rate")
print("=" * 50)

# Clean up memory from previous example
#del pruned_model_1
cleanup_memory()

# Reload model for second example (since first one was modified)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

# Run Example 2
pruned_model_2, stats_2 = example_pruning_by_expansion_rate(model, tokenizer, model_name)

# Add to results
results.append({
    'model': model_name,
    'method': 'Expansion Rate',
    'reduction': stats_2['percentage_reduction'],
    'expansion_rate': stats_2['expansion_rate']
})

print(f"\nExample 2 completed! Reduction: {stats_2['percentage_reduction']:.2f}%")


Starting Example 2: Pruning by Expansion Rate


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-09-09 15:15:43,380 - optipfair.pruning.mlp_glu - INFO - Calculated pruning percentage: 62.16% to achieve expansion rate of 200%


=== Example 2: Pruning Qwen/Qwen2.5-7B-Instruct to 200% expansion rate ===
Original Model: Qwen/Qwen2.5-7B-Instruct
Parameters: 7,615,616,512


Pruning layers: 100%|██████████| 28/28 [00:13<00:00,  2.02it/s]
2025-09-09 15:15:57,226 - optipfair.pruning.mlp_glu - INFO - Updated model config: intermediate_size = 7168
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Pruning Results ---
Original parameters: 7,615,616,512
Pruned parameters: 4,070,381,056
Reduction: 3,545,235,456 parameters (46.55%)
Target expansion rate: 200%
Actual expansion rate: 200.00%

--- Pruned Model Generation ---
Prompt: 'Paris is the capital of'
Generated: Paris is the capital of the 20th century, which is a 1st century. The 3rd century is 4th, and so on, it's 5th. Paris is also 6th and 7th


Example 2 completed! Reduction: 46.55%


## Results Summary

In [19]:
print("\n" + "="*60)
print("PRUNING RESULTS SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Method':<15} {'Reduction':<12} {'Expansion Rate':<15}")
print("-" * 75)

for result in results:
    print(f"{result['model']:<30} {result['method']:<15} {result['reduction']:<12.2f}% {result['expansion_rate']:<15.2f}%")

print(f"\nTotal models tested: {len(results)}")
print("Pruning examples completed successfully!")


PRUNING RESULTS SUMMARY
Model                          Method          Reduction    Expansion Rate 
---------------------------------------------------------------------------
Qwen/Qwen2.5-7B-Instruct       Percentage      14.97       % 422.88         %
Qwen/Qwen2.5-7B-Instruct       Expansion Rate  46.55       % 200.00         %

Total models tested: 2
Pruning examples completed successfully!


---
## ✅ Success! What's Next?

Congratulations! You've successfully pruned a 1B+ parameter model and seen how OptiPFair makes the process simple and straightforward.

If you found this notebook useful, the best way to support the OptiPFair project is by **starring it on GitHub**. Your support is a huge help in boosting the project's visibility and reaching more developers and researchers.

### ➡️ [**Star OptiPFair on GitHub**](https://github.com/peremartra/optipfair)

---
You can also follow my work and new projects on:

* **[LinkedIn](https://www.linkedin.com/in/pere-martra/)**
* **[X / Twitter](https://twitter.com/PereMartra)**